# 04 — Optical-SAR Feature-Level Fusion
**Owner: Person 4**

Priority 3 (mandatory, and the hardest slot). Strategy: **Path A — lightweight feature-level fusion**, not a new large multimodal foundation model.

Pretrained optical encoder + pretrained SAR encoder (frozen or lightly fine-tuned) -> concatenate/cross-attend features -> small trainable fusion head. Train on BigEarthNet-MM (paired Sentinel-2 + Sentinel-1). **Must report an optical-only baseline vs. optical+SAR result** to prove SAR is actually contributing information.

## 1. Environment

In [ ]:
# !pip install torch torchvision rasterio pillow --quiet
import sys, os
sys.path.insert(0, os.path.abspath('..'))


## 2. Load shared configuration

In [ ]:
from src.utils.io_utils import load_config

config = load_config('../configs/config.yaml')
fusion_cfg = config['models']['fusion']
bem_cfg = config['datasets']['bigearthnet_mm']
fusion_cfg, bem_cfg

## 3. Load BigEarthNet-MM paired data
Confirm optical/SAR pairing (same tile ID, same date) before building the training set.

In [ ]:
# from src.preprocessing.dataset_loader import get_dataloader
# bem_loader = get_dataloader(task='fusion', split='train', dataset='bigearthnet_mm', config=config)
# batch = next(iter(bem_loader))
# batch.keys()


## 4. Inspect one optical + SAR pair visually

In [ ]:
# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# axes[0].imshow(sample_optical_rgb); axes[0].set_title('Optical (Sentinel-2)')
# axes[1].imshow(sample_sar_vv, cmap='gray'); axes[1].set_title('SAR (Sentinel-1, VV)')
# plt.show()


## 5. Preprocess optical
Dataset-level (Sentinel-2 bands + percentile normalization) then model-level (fusion encoder's expected input size).

In [ ]:
from src.preprocessing.geotiff_utils import read_image
from src.preprocessing.normalize import preprocess_pipeline

# img = read_image('<path to optical tile>')
# optical_arr = preprocess_pipeline(img.array, config, dataset='bigearthnet_mm',
#                                    model='fusion', modality='optical')


## 6. Preprocess SAR
Uses the `db_scale_minmax` normalization declared for this dataset's SAR bands — implement the dB conversion in `src/preprocessing/normalize.py` if not already done.

In [ ]:
# sar_img = read_image('<path to SAR tile>')
# sar_arr = preprocess_pipeline(sar_img.array, config, dataset='bigearthnet_mm',
#                                model='fusion', modality='sar')


## 7. Load pretrained encoders

In [ ]:
# optical_encoder = load_pretrained_optical_encoder(fusion_cfg['optical_encoder'])
# sar_encoder = load_pretrained_sar_encoder(fusion_cfg['sar_encoder'])


## 8. Extract features

In [ ]:
# optical_features = optical_encoder(optical_arr)
# sar_features = sar_encoder(sar_arr)
# optical_features.shape, sar_features.shape


## 9. Build the fusion module
Keep this small and trainable — concatenation + MLP head, or a lightweight cross-attention block. This is the ONLY part trained from scratch this sprint.

In [ ]:
import torch
import torch.nn as nn

class FusionHead(nn.Module):
    """TODO(Person 4): concatenate or cross-attend optical_features
    and sar_features, then decode into the target output (e.g. a
    built-up/water segmentation mask, or a text-answer head)."""
    def __init__(self, optical_dim: int, sar_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.proj = nn.Linear(optical_dim + sar_dim, hidden_dim)
        # TODO: add the actual task head (classification/segmentation/etc.)

    def forward(self, optical_features, sar_features):
        fused = torch.cat([optical_features, sar_features], dim=-1)
        return self.proj(fused)


## 10. Train the fusion head on BigEarthNet-MM

In [ ]:
# fusion_head = FusionHead(optical_dim=..., sar_dim=...)
# optimizer = torch.optim.AdamW(fusion_head.parameters(), lr=config['training']['lr'])
# for epoch in range(config['training']['epochs']):
#     for batch in bem_loader:
#         loss = train_step(fusion_head, batch, optimizer, config)


## 11. Evaluate: optical-only baseline

In [ ]:
# optical_only_score = evaluate_optical_only(optical_encoder, eval_loader)
# print('Optical-only score:', optical_only_score)


## 12. Evaluate: optical + SAR fusion
**This score should beat the optical-only baseline** — that's the evidence that SAR is contributing real information, not just present for appearances.

In [ ]:
# fusion_score = evaluate_fusion(optical_encoder, sar_encoder, fusion_head, eval_loader)
# print('Optical+SAR score:', fusion_score)
# assert fusion_score >= optical_only_score, 'SAR fusion should not underperform optical-only'


## 13. Save the fusion head checkpoint

In [ ]:
# torch.save(fusion_head.state_dict(), fusion_cfg['checkpoint'])
# print('Saved to', fusion_cfg['checkpoint'])


## 14. Export the inference function
Move the stable implementation into `src/models/fusion_model.py`, including `predict_optical_only()` so the baseline comparison stays reproducible after integration.

In [ ]:
from src.common.schemas import RSModelResult, ResultMetadata, Evidence

# def predict(image_optical, image_sar, query):
#     optical_features = optical_encoder(image_optical)
#     sar_features = sar_encoder(image_sar)
#     fused = fusion_head(optical_features, sar_features)
#     answer, confidence, mask = decode(fused)
#     return RSModelResult(
#         task='fusion', answer=answer, confidence=confidence,
#         evidence=Evidence(mask=mask),
#         metadata=ResultMetadata(model='fusion_v1',
#                                  checkpoint=fusion_cfg['checkpoint'],
#                                  dataset='BigEarthNet-MM',
#                                  backbone='OpticalSAR-FeatureFusion'),
#     ).to_dict()
